In [ ]:
import os
import zipfile

# google.colab only exists in Colab — not when running this notebook in PyCharm/Jupyter
try:
    from google.colab import drive

    drive.mount("/content/drive")
    zip_path = "/content/drive/MyDrive/R_D-CV/Drones_Datasets.zip"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/content")
    os.chdir("/content")
except ImportError:
    # Local run: dataset should already be next to this notebook
    notebook_dir = os.path.dirname(os.path.abspath("train_model_drones_dataset.ipynb"))
    os.chdir(notebook_dir)
    assert os.path.exists("Drones_Datasets/data.yaml"), (
        "Drones_Datasets/data.yaml not found. Open the notebook from model/ "
        "or put Drones_Datasets next to it."
    )
    print("Not in Colab — using local Drones_Datasets/")

In [ ]:
import torch
from ultralytics import YOLO

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — Runtime → Change runtime type → T4 GPU, then re-run all cells")

In [ ]:
# Faster Colab settings: small model, fewer epochs, slightly smaller images
model = YOLO("yolo11n.pt")

train_results = model.train(
    data="Drones_Datasets/data.yaml",
    batch=16,       # raise if GPU has headroom; lower to 8 if OOM
    epochs=30,      # 50 is slow; raise later if needed
    imgsz=416,      # 640 is slower; use 640 for final quality
    project="runs/detect",
    name="drones_datasets_yolov11",
    exist_ok=True,  # reuse same folder instead of drones_datasets_yolov112, ...
    save=True,
    patience=10,
    save_period=-1,
    val=True,
    workers=2,
    cache=False,
    amp=True,       # mixed precision — faster on GPU
    device=0 if torch.cuda.is_available() else "cpu",
)
print("Results saved to:", train_results.save_dir)
print("Metrics CSV:", f"{train_results.save_dir}/results.csv")


In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

print("cwd:", os.getcwd())

candidates = []
if "train_results" in globals() and getattr(train_results, "save_dir", None):
    candidates.append(os.path.join(str(train_results.save_dir), "results.csv"))
if "model" in globals() and getattr(model, "trainer", None) is not None:
    candidates.append(os.path.join(str(model.trainer.save_dir), "results.csv"))

search_roots = [".", "/content", "/content/drive/MyDrive"]
for root in search_roots:
    if os.path.isdir(root):
        candidates.extend(glob.glob(os.path.join(root, "**/results.csv"), recursive=True))

existing = [p for p in candidates if os.path.isfile(p)]
assert existing, (
    "No results.csv found.\n"
    "Colab deletes /content/runs when the runtime restarts — re-run the training cell first.\n"
    f"cwd={os.getcwd()}\n"
    f"runs exists: {os.path.isdir('runs')}\n"
    f"detect dirs: {glob.glob('runs/detect/*')}"
)
results_csv = max(existing, key=os.path.getmtime)
print("Using:", results_csv)

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()


def plot_history(df):
    """Keras-style history plot, adapted for YOLO detect metrics (no dfl_loss)."""
    epochs = df["epoch"]
    # Detection has no single accuracy — use box+cls loss and val mAP / precision / recall
    train_loss = df["train/box_loss"] + df["train/cls_loss"]
    val_loss = df["val/box_loss"] + df["val/cls_loss"]

    plt.figure(figsize=(12, 4))

    plt.subplot(121)
    plt.plot(epochs, train_loss, ".-", epochs, val_loss, ".-")
    plt.grid(True)
    plt.xlabel("epochs")
    plt.ylabel("loss (box + cls)")
    plt.legend(["Train", "Validation"])

    plt.subplot(122)
    plt.plot(epochs, df["metrics/mAP50(B)"], ".-", label="mAP50")
    plt.plot(epochs, df["metrics/mAP50-95(B)"], ".-", label="mAP50-95")
    plt.plot(epochs, df["metrics/precision(B)"], ".-", label="precision")
    plt.plot(epochs, df["metrics/recall(B)"], ".-", label="recall")
    plt.grid(True)
    plt.xlabel("epochs")
    plt.ylabel("metrics")
    plt.legend()

    plt.tight_layout()
    plt.show()

    print("Train loss (box+cls)     ", float(train_loss.iloc[-1]))
    print("Validation loss (box+cls)", float(val_loss.iloc[-1]))
    print("mAP50                    ", float(df["metrics/mAP50(B)"].iloc[-1]))
    print("mAP50-95                 ", float(df["metrics/mAP50-95(B)"].iloc[-1]))
    print("Precision                ", float(df["metrics/precision(B)"].iloc[-1]))
    print("Recall                   ", float(df["metrics/recall(B)"].iloc[-1]))


plot_history(df)


In [1]:
import matplotlib.pyplot as plt
import os
import random
import cv2

model = YOLO("runs/detect/drones_datasets_yolov11/weights/best.pt")

images_path = "Drones_Datasets/train/images"

fig, ax = plt.subplots(4, 2, figsize=(15, 16))
ax = ax.ravel()

img_files = [
    f for f in os.listdir(images_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
]

for i in range(8):
    img_file_name = random.choice(img_files)
    img_path = os.path.join(images_path, img_file_name)

    img = cv2.imread(img_path)
    if img is None:
        continue

    results = model(img)[0]

    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ax[i].imshow(img)
    ax[i].axis("off")

plt.show()

NameError: name 'YOLO' is not defined